**Decorators**
A function that takes input a function and returns a function.

In [ ]:
import time
from typing import Callable

def log_time(func:Callable) -> Callable:  
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs) #calls the original function
        end_time = time.time()
        print(f"Function {func.__name__} took {end_time - start_time:.4f} seconds to execute.")
        return result
    return wrapper

@log_time
def llm_chat_response(prompt:str) -> str:
    time.sleep(1)  # Simulate a delay for the LLM response
    return f"Response to: {prompt}"

@log_time
def llm_chat_response_with_error(prompt:str) -> str:
    time.sleep(1)  # Simulate a delay for the LLM response
    raise Exception("Simulated LLM error")

@log_time
def text_embedding(text:str) -> list[float]:
    time.sleep(0.5)  # Simulate a delay for the embedding generation
    return [0.1, 0.2, 0.3]  # Dummy embedding vector


llm_chat_response("Hello, how are you?")
try:
    llm_chat_response_with_error("This will raise an error")
except Exception as e:
    print(f"Caught an exception: {e}")
    
text_embedding("Generate embedding for this text.")


Function llm_chat_response took 1.0051 seconds to execute.
Caught an exception: Simulated LLM error
Function text_embedding took 0.5029 seconds to execute.


[0.1, 0.2, 0.3]

In [3]:
def retry(func:Callable, retries:int=3) -> Callable:
    
    def inner_function(*args, **kwargs):
        for attempt in range(retries):
            try:
                return func(*args, **kwargs)
            except Exception as e:
                print(f"Attempt '{func.__name__}' {attempt + 1} failed with error: {e}")
                if attempt == retries - 1:
                    raise
    return inner_function

@retry
def llm_chat_response_with_retry(prompt:str) -> str:
    time.sleep(1)  # Simulate a delay for the LLM response
    if prompt == "fail":
        raise Exception("Simulated LLM error for retry")
    return f"Response to: {prompt}"


#llm_chat_response_with_retry("fail")  # This will trigger retries and eventually raise an exception

llm_chat_response_with_retry("Hello, retry mechanism!")

print(llm_chat_response_with_retry.__name__)  # This should print "inner_function" because the original function is wrapped by the retry decorator.




inner_function


Use functools.wraps to retain calling functions metadata

In [4]:
from functools import wraps
def retry(func:Callable, retries:int=3) -> Callable:
    @wraps(func) #retains the original function's metadata
    def inner_function(*args, **kwargs):
        for attempt in range(retries):
            try:
                return func(*args, **kwargs)
            except Exception as e:
                print(f"Attempt '{func.__name__}' {attempt + 1} failed with error: {e}")
                if attempt == retries - 1:
                    raise
    return inner_function

@retry
def llm_chat_response_with_retry(prompt:str) -> str:
    time.sleep(1)  # Simulate a delay for the LLM response
    if prompt == "fail":
        raise Exception("Simulated LLM error for retry")
    return f"Response to: {prompt}"

print(llm_chat_response_with_retry.__name__) 

llm_chat_response_with_retry


when we try to configure retries count , it fails

In [5]:
from functools import wraps
def retry(func:Callable, retries:int=3) -> Callable:
    @wraps(func) #retains the original function's metadata
    def inner_function(*args, **kwargs):
        for attempt in range(retries):
            try:
                return func(*args, **kwargs)
            except Exception as e:
                print(f"Attempt '{func.__name__}' {attempt + 1} failed with error: {e}")
                if attempt == retries - 1:
                    raise
    return inner_function

@retry(retries=3)
def llm_chat_response_with_retry(prompt:str) -> str:
    time.sleep(1)  # Simulate a delay for the LLM response
    if prompt == "fail":
        raise Exception("Simulated LLM error for retry")
    return f"Response to: {prompt}"

llm_chat_response_with_retry("fail") 

TypeError: retry() missing 1 required positional argument: 'func'

In [6]:
from functools import wraps
from typing import Callable

def retry(retries:int=3) -> Callable:
    def decorator_function(func: Callable) -> Callable:
        @wraps(func) #retains the original function's metadata
        def inner_function(*args, **kwargs):
            for attempt in range(retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Attempt '{func.__name__}' {attempt + 1} failed with error: {e}")
                    if attempt == retries - 1:
                        raise
        return inner_function
    return decorator_function

@retry(retries=2)
def llm_chat_response_with_retry(prompt:str) -> str:
    time.sleep(1)  # Simulate a delay for the LLM response
    if prompt == "fail":
        raise Exception("Simulated LLM error for retry")
    return f"Response to: {prompt}"

llm_chat_response_with_retry("fail") 

Attempt 'llm_chat_response_with_retry' 1 failed with error: Simulated LLM error for retry
Attempt 'llm_chat_response_with_retry' 2 failed with error: Simulated LLM error for retry


Exception: Simulated LLM error for retry